# NowcastMY — Autonomous Macroeconomic Nowcasting

An LLM agent autonomously improves a GDP nowcasting model for Malaysia, 
using open data from DOSM and the autoresearch-lite experiment loop.

**How it works:**
1. `prepare_data.py` downloads DOSM parquet files and builds a merged panel
2. `baselines.py` establishes the AR(1) RMSE to beat
3. The agent loop reads `nowcast.py`, sends it to an LLM, applies the proposed edit, runs the model, and keeps or reverts based on RMSE
4. Repeat 100–200 times

**Requirements:**
- An OpenRouter API key (free tier works: https://openrouter.ai)
- All other data is free, open, and requires no authentication

**Runtime:** CPU is sufficient (no GPU needed). Models fit in <1 second.

---
## Cell 0: Configuration
Fill in your API key and run tag.

In [ ]:
#@title Configuration { display-mode: "form" }

#@markdown ### Required: OpenRouter API key
#@markdown Get one free at https://openrouter.ai/keys
OPENROUTER_API_KEY = ""  #@param {type:"string"}

#@markdown ### LLM model to use (free tier — paste any OpenRouter model ID)
#@markdown Verified free models: `nousresearch/hermes-3-llama-3.1-405b:free`, `meta-llama/llama-3.3-70b-instruct:free`, `mistralai/mistral-small-3.1-24b-instruct:free`, `qwen/qwen3-235b-a22b:free`, `openrouter/free`
LLM_MODEL = "nousresearch/hermes-3-llama-3.1-405b:free"  #@param {type:"string"}

#@markdown ### Run tag (for git branch naming)
RUN_TAG = "run1"  #@param {type:"string"}

#@markdown ### Max experiments to run
MAX_EXPERIMENTS = 100  #@param {type:"integer"}

#@markdown ### Data directory
DATA_DIR = "./data"  #@param {type:"string"}

assert OPENROUTER_API_KEY, "Please set your OpenRouter API key above"
print(f"Config OK: model={LLM_MODEL}, tag={RUN_TAG}, max_experiments={MAX_EXPERIMENTS}")

---
## Cell 1: Setup
Install dependencies and download DOSM data.

In [ ]:
%%time
# Install dependencies
!pip install -q pandas pyarrow fastparquet scikit-learn xgboost lightgbm statsmodels matplotlib requests

# Upload the project files if not present
import os

REQUIRED_FILES = ['prepare_data.py', 'baselines.py', 'nowcast.py', 'program_nowcast.md']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]

if missing:
    print(f"Missing files: {missing}")
    print("Please upload them:")
    from google.colab import files
    uploaded = files.upload()
    # Verify
    still_missing = [f for f in REQUIRED_FILES if not os.path.exists(f)]
    assert not still_missing, f"Still missing: {still_missing}"

print("All project files present.")

# Initialize git repo for experiment tracking
if not os.path.exists('.git'):
    !git init
    !git config user.email "nowcastmy@colab"
    !git config user.name "NowcastMY Agent"
    !git add -A
    !git commit -m "initial commit"
    print("Git repo initialized.")

# Download DOSM data
print("\n" + "="*60)
print("Downloading DOSM data...")
print("="*60)
!python prepare_data.py --cache-dir {DATA_DIR}

---
## Cell 2: Baselines
Run AR(1), AR(2), random walk baselines. This gives us the RMSE to beat.

In [ ]:
!python baselines.py --data-dir {DATA_DIR}

# Store baseline RMSE for the agent loop
import pandas as pd
baselines_df = pd.read_csv(f"{DATA_DIR}/baselines_results.csv")
ar1_row = baselines_df[(baselines_df['model'] == 'AR(1)') & (baselines_df['variant'] == 'full')]
BASELINE_AR1_RMSE = ar1_row.iloc[0]['rmse']
print(f"\n>>> AR(1) RMSE to beat: {BASELINE_AR1_RMSE:.4f}")

---
## Cell 3: Agent Helpers
LLM API wrapper, diff parser, git commit/revert, results logger.

In [ ]:
import re
import json
import time
import subprocess
import requests
from datetime import datetime
from pathlib import Path

# ── LLM API ──────────────────────────────────────────────────────────

def call_llm(prompt: str, max_retries: int = 3) -> str:
    """Call OpenRouter API with retry and backoff."""
    for attempt in range(max_retries):
        try:
            resp = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
                    "Content-Type": "application/json",
                },
                json={
                    "model": LLM_MODEL,
                    "messages": [{"role": "user", "content": prompt}],
                    "max_tokens": 4096,
                    "temperature": 0.7,
                },
                timeout=120,
            )
            resp.raise_for_status()
            data = resp.json()
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            wait = 2 ** (attempt + 1)
            print(f"  LLM call failed (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)
    return ""


# ── Diff Parser ──────────────────────────────────────────────────────

def parse_diff(response: str):
    """
    Parse the LLM response for DESCRIPTION / OLD / NEW format.
    Returns (description, old_code, new_code) or None if parsing fails.
    """
    # Try to find DESCRIPTION, OLD, NEW blocks
    desc_match = re.search(r'DESCRIPTION[:\s]*\n(.*?)(?=\nOLD[:\s]*\n|$)', response, re.DOTALL)
    old_match = re.search(r'OLD[:\s]*\n```[^\n]*\n(.*?)```', response, re.DOTALL)
    new_match = re.search(r'NEW[:\s]*\n```[^\n]*\n(.*?)```', response, re.DOTALL)

    # Fallback: try without code fences
    if not old_match:
        old_match = re.search(r'OLD[:\s]*\n(.*?)(?=\nNEW[:\s]*\n)', response, re.DOTALL)
    if not new_match:
        new_match = re.search(r'NEW[:\s]*\n(.*?)(?=\n(?:DESCRIPTION|$))', response, re.DOTALL)

    if not all([desc_match, old_match, new_match]):
        return None

    description = desc_match.group(1).strip()
    old_code = old_match.group(1).rstrip('\n')
    new_code = new_match.group(1).rstrip('\n')

    return description, old_code, new_code


def apply_diff(filepath: str, old_code: str, new_code: str) -> bool:
    """Apply a string replacement diff to a file. Returns True if successful."""
    try:
        content = open(filepath).read()
        if old_code not in content:
            # Try with stripped whitespace
            old_stripped = old_code.strip()
            if old_stripped in content:
                content = content.replace(old_stripped, new_code.strip(), 1)
            else:
                print(f"  ERROR: OLD block not found in {filepath}")
                return False
        else:
            content = content.replace(old_code, new_code, 1)
        open(filepath, 'w').write(content)
        return True
    except Exception as e:
        print(f"  ERROR applying diff: {e}")
        return False


# ── Git Helpers ───────────────────────────────────────────────────────

def git_commit(message: str):
    """Stage and commit all changes."""
    subprocess.run(["git", "add", "-A"], capture_output=True)
    subprocess.run(["git", "commit", "-am", message], capture_output=True)


def git_revert():
    """Revert nowcast.py to the last committed version."""
    subprocess.run(["git", "checkout", "--", "nowcast.py"], capture_output=True)


def git_short_hash() -> str:
    """Get current short commit hash."""
    r = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True)
    return r.stdout.strip()


# ── Run Experiment ────────────────────────────────────────────────────

def run_nowcast() -> dict:
    """
    Run nowcast.py and parse the output metrics.
    Returns dict with rmse_vs_ar1, coverage, and full stdout.
    """
    result = subprocess.run(
        ["python", "nowcast.py", "--data-dir", DATA_DIR],
        capture_output=True, text=True, timeout=120,
    )
    stdout = result.stdout
    stderr = result.stderr

    # Parse metrics
    rmse_match = re.search(r'RMSE_VS_AR1=([\d.]+)', stdout)
    cov_match = re.search(r'COVERAGE_90PCT=([\d.nan]+)', stdout)

    if rmse_match:
        rmse_vs_ar1 = float(rmse_match.group(1))
    else:
        rmse_vs_ar1 = 999.0  # crash

    if cov_match and cov_match.group(1) != 'nan':
        coverage = float(cov_match.group(1))
    else:
        coverage = float('nan')

    return {
        "rmse_vs_ar1": rmse_vs_ar1,
        "coverage": coverage,
        "stdout": stdout,
        "stderr": stderr,
        "crashed": rmse_vs_ar1 >= 999.0,
    }


# ── Results Logger ────────────────────────────────────────────────────

TSV_PATH = Path(DATA_DIR) / "results.tsv"

def log_result(exp_id, description, result, status, config=None):
    """Append a row to results.tsv."""
    config = config or {}
    row = {
        "experiment_id": exp_id,
        "timestamp": datetime.now().isoformat(),
        "description": description,
        "nowcast_mode": "direct",
        "features_used": config.get("features", ""),
        "model_type": config.get("model_type", ""),
        "n_lags": config.get("n_lags", ""),
        "ragged_edge_method": "forward_fill",
        "rmse_m1": f"{result['rmse_vs_ar1']:.4f}" if not result['crashed'] else "crash",
        "rmse_m2": f"{result['rmse_vs_ar1']:.4f}" if not result['crashed'] else "crash",
        "rmse_m3": f"{result['rmse_vs_ar1']:.4f}" if not result['crashed'] else "crash",
        "rmse_avg": f"{result['rmse_vs_ar1']:.4f}" if not result['crashed'] else "crash",
        "mae": "",
        "rmse_vs_ar1": f"{result['rmse_vs_ar1']:.6f}" if not result['crashed'] else "crash",
        "coverage_90pct": f"{result['coverage']:.4f}" if not result['crashed'] and result['coverage'] == result['coverage'] else "nan",
        "status": status,
    }
    row_str = "\t".join(str(v) for v in row.values())
    with open(TSV_PATH, "a") as f:
        f.write(row_str + "\n")


def get_results_history() -> str:
    """Read results.tsv as formatted string for the LLM prompt."""
    if not TSV_PATH.exists():
        return "No experiments yet."
    content = open(TSV_PATH).read()
    lines = content.strip().split("\n")
    # Return last 20 experiments max (avoid context overflow)
    if len(lines) > 21:
        return "\n".join([lines[0]] + lines[-20:])
    return content


# ── Prompt Builder ────────────────────────────────────────────────────

def build_prompt(experiment_id: int) -> str:
    """Build the LLM prompt with current code, history, and instructions."""
    nowcast_code = open("nowcast.py").read()
    program_instructions = open("program_nowcast.md").read()
    history = get_results_history()

    prompt = f"""You are an autonomous research agent running macroeconomic nowcasting experiments.

## Instructions
{program_instructions}

## Current nowcast.py
```python
{nowcast_code}
```

## Experiment history (results.tsv)
```
{history}
```

## Your task
This is experiment #{experiment_id}. Based on the instructions and history above, propose ONE change to nowcast.py.

Respond in EXACTLY this format:

DESCRIPTION
<one line describing what you're trying and why>

OLD
```python
<exact code block to replace — must match nowcast.py exactly>
```

NEW
```python
<replacement code block>
```

IMPORTANT:
- The OLD block must be an EXACT substring of the current nowcast.py (copy-paste, don't paraphrase)
- Change only ONE thing per experiment
- The last two lines of stdout must still be RMSE_VS_AR1=... and COVERAGE_90PCT=...
- Follow the phased strategy: Phase A (exp 1-10), Phase B (11-20), Phase C (21-30), Phase D (31-40), Phase E (41+)
"""
    return prompt


print("Agent helpers loaded.")
print(f"  LLM model: {LLM_MODEL}")
print(f"  Results: {TSV_PATH}")

---
## Cell 4: Agent Loop
The main autonomous experiment loop. Interrupt the kernel to stop.

In [ ]:
import numpy as np

# Create git branch for this run
!git checkout -b nowcast/{RUN_TAG} 2>/dev/null || git checkout nowcast/{RUN_TAG}

# ── Run baseline first ──────────────────────────────────────────────
print("Running baseline (experiment 0)...")
baseline_result = run_nowcast()
best_rmse = baseline_result["rmse_vs_ar1"]
print(f"  Baseline RMSE_VS_AR1 = {best_rmse:.4f}")
print(f"  Baseline COVERAGE    = {baseline_result['coverage']:.4f}")
git_commit("experiment 0: baseline")
log_result(0, "baseline: IPI only Ridge", baseline_result, "keep",
           {"features": "ipi", "model_type": "ridge", "n_lags": 1})

# ── Main loop ────────────────────────────────────────────────────────
consecutive_failures = 0
MAX_CONSECUTIVE_FAILURES = 5

for exp_id in range(1, MAX_EXPERIMENTS + 1):
    print(f"\n{'='*60}")
    print(f"  Experiment {exp_id}/{MAX_EXPERIMENTS}")
    print(f"  Best RMSE_VS_AR1 so far: {best_rmse:.4f}")
    print(f"{'='*60}")

    # 1. Build prompt
    prompt = build_prompt(exp_id)

    # 2. Call LLM
    print("  Calling LLM...", end=" ", flush=True)
    response = call_llm(prompt)
    if not response:
        print("EMPTY response")
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping.")
            break
        continue
    print("OK")

    # 3. Parse diff
    parsed = parse_diff(response)
    if parsed is None:
        print("  Could not parse DESCRIPTION/OLD/NEW from response.")
        print(f"  Response preview: {response[:200]}...")
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping.")
            break
        continue

    description, old_code, new_code = parsed
    print(f"  Description: {description}")

    # 4. Apply diff
    if not apply_diff("nowcast.py", old_code, new_code):
        print("  Diff application failed. Reverting.")
        git_revert()
        consecutive_failures += 1
        if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
            print(f"  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping.")
            break
        continue

    # 5. Git commit
    git_commit(f"experiment {exp_id}: {description[:60]}")

    # 6. Run experiment
    print("  Running nowcast.py...", end=" ", flush=True)
    try:
        result = run_nowcast()
    except subprocess.TimeoutExpired:
        print("TIMEOUT")
        result = {"rmse_vs_ar1": 999.0, "coverage": float('nan'), "stdout": "", "stderr": "timeout", "crashed": True}
    except Exception as e:
        print(f"ERROR: {e}")
        result = {"rmse_vs_ar1": 999.0, "coverage": float('nan'), "stdout": "", "stderr": str(e), "crashed": True}

    # 7. Evaluate
    if result["crashed"]:
        print(f"CRASHED")
        if result["stderr"]:
            print(f"  stderr (last 200): {result['stderr'][-200:]}")
        status = "crash"
        git_revert()
        git_commit(f"revert experiment {exp_id} (crash)")
        consecutive_failures += 1
    elif result["rmse_vs_ar1"] < best_rmse:
        improvement = best_rmse - result["rmse_vs_ar1"]
        print(f"IMPROVED! {result['rmse_vs_ar1']:.4f} (was {best_rmse:.4f}, delta={improvement:.4f})")
        best_rmse = result["rmse_vs_ar1"]
        status = "keep"
        consecutive_failures = 0
    else:
        print(f"No improvement: {result['rmse_vs_ar1']:.4f} >= {best_rmse:.4f}")
        status = "discard"
        git_revert()
        git_commit(f"revert experiment {exp_id} (no improvement)")
        consecutive_failures = 0  # not a failure, just not better

    # 8. Log
    log_result(exp_id, description, result, status)

    if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
        print(f"\n  {MAX_CONSECUTIVE_FAILURES} consecutive failures. Stopping.")
        break

    # Brief pause to respect API rate limits
    time.sleep(1)

print(f"\n{'='*60}")
print(f"  Agent loop finished")
print(f"  Total experiments: {exp_id}")
print(f"  Best RMSE_VS_AR1: {best_rmse:.4f}")
print(f"{'='*60}")

---
## Cell 5: Analysis
Plot RMSE convergence, feature importance, and best model summary.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Load results
tsv = pd.read_csv(f"{DATA_DIR}/results.tsv", sep="\t")
print(f"Total experiments logged: {len(tsv)}")
print(f"Kept: {(tsv['status']=='keep').sum()}, Discarded: {(tsv['status']=='discard').sum()}, Crashed: {(tsv['status']=='crash').sum()}")

# Filter to numeric RMSE values
tsv_valid = tsv[tsv['rmse_vs_ar1'] != 'crash'].copy()
tsv_valid['rmse_vs_ar1'] = tsv_valid['rmse_vs_ar1'].astype(float)
tsv_valid['experiment_id'] = tsv_valid['experiment_id'].astype(int)

# ── Plot 1: RMSE convergence ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('NowcastMY — Agent Loop Results', fontsize=14, fontweight='bold')

ax = axes[0]
kept = tsv_valid[tsv_valid['status'] == 'keep']
discarded = tsv_valid[tsv_valid['status'] == 'discard']
ax.scatter(discarded['experiment_id'], discarded['rmse_vs_ar1'],
           c='#CCCCCC', s=20, label='Discarded', zorder=2)
ax.scatter(kept['experiment_id'], kept['rmse_vs_ar1'],
           c='#2E75B6', s=40, label='Kept', zorder=3)

# Best-so-far line
best_so_far = tsv_valid['rmse_vs_ar1'].expanding().min()
ax.plot(tsv_valid['experiment_id'], best_so_far, 'r-', linewidth=1.5, label='Best so far', zorder=4)
ax.axhline(1.0, color='gray', linestyle='--', linewidth=0.8, label='AR(1) baseline')
ax.axhline(0.8, color='green', linestyle=':', linewidth=0.8, label='PRD target (0.80)')
ax.set_xlabel('Experiment')
ax.set_ylabel('RMSE / AR(1)')
ax.set_title('RMSE ratio convergence')
ax.legend(fontsize=8)

# ── Plot 2: Status distribution ──────────────────────────────────────
ax = axes[1]
status_counts = tsv['status'].value_counts()
colors = {'keep': '#2E75B6', 'discard': '#CCCCCC', 'crash': '#E24B4A', 'baseline': '#27AE60'}
bars = ax.bar(status_counts.index, status_counts.values,
              color=[colors.get(s, '#888') for s in status_counts.index])
ax.set_ylabel('Count')
ax.set_title('Experiment outcomes')
for bar, val in zip(bars, status_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', fontsize=11)

# ── Plot 3: Coverage calibration ─────────────────────────────────────
ax = axes[2]
if 'coverage_90pct' in tsv_valid.columns:
    cov_valid = tsv_valid[tsv_valid['coverage_90pct'] != 'nan'].copy()
    if len(cov_valid) > 0:
        cov_valid['coverage_90pct'] = cov_valid['coverage_90pct'].astype(float)
        ax.scatter(cov_valid['experiment_id'], cov_valid['coverage_90pct'],
                   c='#2E75B6', s=20)
        ax.axhline(0.90, color='green', linestyle='--', linewidth=0.8, label='Target (90%)')
        ax.axhspan(0.85, 0.95, alpha=0.1, color='green', label='Acceptable range')
        ax.set_ylim(0.5, 1.05)
        ax.legend(fontsize=8)
ax.set_xlabel('Experiment')
ax.set_ylabel('Coverage')
ax.set_title('90% interval calibration')

plt.tight_layout()
plt.savefig(f'{DATA_DIR}/agent_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Best model summary ───────────────────────────────────────────────
if len(kept) > 0:
    best_row = kept.loc[kept['rmse_vs_ar1'].idxmin()]
    print(f"\n{'='*60}")
    print(f"  BEST MODEL")
    print(f"{'='*60}")
    print(f"  Experiment:    {int(best_row['experiment_id'])}")
    print(f"  Description:   {best_row['description']}")
    print(f"  RMSE_VS_AR1:   {best_row['rmse_vs_ar1']:.4f}")
    print(f"  Model:         {best_row.get('model_type', 'N/A')}")
    print(f"  Features:      {best_row.get('features_used', 'N/A')}")
    print(f"  Status:        {'BEATS AR(1)' if best_row['rmse_vs_ar1'] < 1.0 else 'Does not beat AR(1)'}")
    if best_row['rmse_vs_ar1'] < 0.80:
        print(f"  >>> PRD TARGET MET (< 0.80)! <<<")
    print(f"{'='*60}")

---
## Cell 6: Export
Download the best model and results.

In [ ]:
from google.colab import files

# Package results
!cp nowcast.py {DATA_DIR}/best_nowcast.py
!tar czf nowcastmy_results.tar.gz {DATA_DIR}/results.tsv {DATA_DIR}/baselines_results.csv {DATA_DIR}/best_nowcast.py {DATA_DIR}/agent_results.png {DATA_DIR}/panel_quarterly.parquet 2>/dev/null

print("Packaged results:")
!ls -lh nowcastmy_results.tar.gz

files.download('nowcastmy_results.tar.gz')
print("\nDownload triggered. You can also download individual files:")
print(f"  - {DATA_DIR}/results.tsv (full experiment log)")
print(f"  - {DATA_DIR}/best_nowcast.py (best model)")
print(f"  - {DATA_DIR}/agent_results.png (convergence plot)")